In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Mount Drive + clone upstream moment_detr + extract features to LOCAL SSD.
# Run once at the top of any modular notebook. Idempotent within a session.

import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# --- 1. Drive --------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), (
    f"Upload your repo folder to {REPO_DIR} first. "
    f"Found nothing at that path."
)

# --- 2. Put repo on sys.path (so `from src.*` works) -----------------------
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 2b. Symlink data dirs INTO the repo so relative paths work ------------
# Notebooks reference 'processed/X.jsonl', 'outputs/Y.png' etc., assuming
# those dirs live INSIDE the repo. In your Drive layout they sit ALONGSIDE
# the repo. Symlinks bridge that gap.
DATA_DIRS_TO_LINK = [
    'processed', 'outputs',
    'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
    'qvh_frames_within_neg', 'qvh_frames_multi',
    'checkpoints',
]
for d in DATA_DIRS_TO_LINK:
    src = DRIVE_ROOT / d
    dst = REPO_DIR / d
    if not src.exists():
        continue
    if dst.exists() or dst.is_symlink():
        continue
    try:
        os.symlink(src, dst)
        print(f"  symlinked  {d}/  ->  {src}")
    except OSError as e:
        print(f"  WARN: couldn't symlink {d}: {e}")

# --- 3. Extract M1 features to LOCAL SSD (NOT to Drive) --------------------
# Drive cannot reliably store 19k small .npz files; we extract every session.
ARCHIVE   = DRIVE_ROOT / 'moment_detr_features.tar.gz'
LOCAL_OUT = Path('/content/features_local')


def _find_feature_dirs(root):
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text


CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if LOCAL_OUT.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] already extracted this session.")
else:
    assert ARCHIVE.exists(), f"Archive missing: {ARCHIVE}"
    if LOCAL_OUT.exists():
        shutil.rmtree(LOCAL_OUT)
    LOCAL_OUT.mkdir(parents=True)
    print(f"[features] extracting {ARCHIVE.stat().st_size/1e9:.2f} GB -> {LOCAL_OUT}")
    t0 = time.time()
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(LOCAL_OUT)
    print(f"[features] extracted in {time.time()-t0:.0f}s")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, "Feature dirs not found after extraction."

# --- 4. Clone upstream Moment-DETR (used by src.models.moment_detr) --------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# --- 5. Expose path variables for downstream cells -------------------------
# Any cell that reads configs/*.yaml can override:
#   cfg['paths']['feature_dir']   = str(CLIP_FEAT_DIR)
#   cfg['paths']['text_feat_dir'] = str(TEXT_FEAT_DIR)
# (configs already point at /content/features_local/... so usually no override needed.)

# --- 6. Verification report -------------------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")
print()
print("Bootstrap done. You can now run the rest of the notebook.")


# Day-1 evening smoke test — does the consistency formula even work?

Owner: Person D.

Goal: Plug **ground-truth queries** into the scorer on the 600-pair eval set.
If AUC with GT queries ≥ 0.75, the formula is sound and M2 quality will only modulate it.
If AUC with GT queries ≈ 0.5, the formula is broken — investigate before training M2.

This is risk **R1** from FINAL_DESIGN section 9. Must pass before Day 2.

In [1]:
import sys, json
from pathlib import Path
PROJECT = Path('.').resolve()
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import numpy as np
import torch
from tqdm.auto import tqdm

from src.utils.io import load_jsonl, save_jsonl, load_yaml
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig
from src.eval.metrics import summarize_separation, confusion_at_threshold

cfg = load_yaml('configs/consistency.yaml')
ScoringConfig.__init__.__defaults__  # type check
scoring_cfg = ScoringConfig(alpha=cfg['alpha'], tau=cfg['tau'],
                             top_p=cfg['top_p'],
                             chunk_window=cfg['chunk_window'],
                             chunk_stride=cfg['chunk_stride'],
                             damp_with_fg=cfg['damp_with_fg'])

m1 = MomentDETRFeaturePredictor(
    ckpt_path=cfg['paths']['m1_ckpt'],
    repo_root=cfg['paths']['repo_root'],
)
scorer = ConsistencyScorer(m1, cfg['paths']['feature_dir'], scoring_cfg)
print('Scorer ready.')

ModuleNotFoundError: No module named 'src'

In [ ]:
pairs = load_jsonl('processed/consistency_eval.jsonl')
print(f'Eval pairs: {len(pairs)}')

# Use the THUMB's query (positives) or the THUMB's query against a DIFFERENT video
# (negatives). The thumb_query field is the GT for the thumbnail-side anchor.
results = []
for r in tqdm(pairs):
    out = scorer.score(
        thumbnail_path=r['frame_path'],
        vid=r['video_vid'],
        queries=[r['thumb_query']],
    )
    results.append({
        'pair_id': r['pair_id'],
        'label':   r['label'],
        'score':   out['C'],
        'queries': out['queries'],
    })

Path('outputs/predictions').mkdir(parents=True, exist_ok=True)
save_jsonl(results, 'outputs/predictions/consistency_gt_query.jsonl')

In [ ]:
scores = [r['score'] for r in results]
labels = [r['label'] for r in results]
summary = summarize_separation(scores, labels)
print(json.dumps(summary, indent=2))

median = float(np.median(scores))
print('\nConfusion at threshold = median:')
print(json.dumps(confusion_at_threshold(scores, labels, median), indent=2))

import matplotlib.pyplot as plt
import numpy as np
pos = [s for s, l in zip(scores, labels) if l == 1]
neg = [s for s, l in zip(scores, labels) if l == 0]
plt.hist(pos, bins=30, alpha=0.5, label=f'positive (n={len(pos)})')
plt.hist(neg, bins=30, alpha=0.5, label=f'negative (n={len(neg)})')
plt.legend(); plt.xlabel('C(T, V)'); plt.title(f"GT-query smoke test  AUC={summary['auc']:.3f}")
plt.savefig('outputs/figures/score_distribution_gt.png', dpi=150, bbox_inches='tight')
plt.show()